# 09 — Full mini-pipeline in PyTorch (project-style)

We've built every piece in NumPy. Now we'll wire up a **complete** PyTorch pipeline that mirrors what the project's main code does, side-by-side with the from-scratch equivalents.

## Roadmap
1. `Dataset` + `DataLoader` — replaces our manual batching
2. `nn.Module` — replaces our class with manual `.forward()`
3. **Autograd** — replaces 200 lines of manual backprop with one `.backward()` call
4. `Optimizer` — Adam in one line
5. Full training loop end-to-end on a CarDD-style multi-label problem
6. Comparison table: lines of code, training time, final accuracy


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Dataset class

Compare:

**From-scratch (NB 02):** we sliced raw NumPy arrays in a Python loop.
**PyTorch:** subclass `Dataset`, implement `__len__` and `__getitem__`, hand to `DataLoader` for batching + shuffling + multi-worker loading.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

class CIFARMultiClass(Dataset):
    """Wrap torchvision's CIFAR-10 and apply our own transform."""

    def __init__(self, root, train=True, n=None):
        self.ds = datasets.CIFAR10(root=root, train=train, download=True)
        self.n = n if n is not None else len(self.ds)
        self.tfm = transforms.Compose([
            transforms.ToTensor(),                               # PIL → CHW float [0,1]
            transforms.Normalize(mean=[0.49, 0.48, 0.45],
                                  std= [0.25, 0.24, 0.26]),
        ])

    def __len__(self):  return self.n
    def __getitem__(self, idx):
        img, label = self.ds[idx]
        return self.tfm(img), label

train_ds = CIFARMultiClass("./_cifar_cache", train=True, n=2000)
test_ds  = CIFARMultiClass("./_cifar_cache", train=False, n=500)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=0)
print("train batches:", len(train_loader), " test batches:", len(test_loader))


## 2. `nn.Module` — a class with weight tracking built in

Same TinyCNN we built in NB 04, with one difference: PyTorch automatically tracks every `nn.*` member as a parameter so optimisers can find them all.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)                  # NB 06 idea — drop-in
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)
        self.fc    = nn.Linear(32 * 8 * 8, num_classes)
        self.drop  = nn.Dropout(0.25)                    # NB 06 idea

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.bn1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.bn2(self.conv2(x))), 2)
        x = self.drop(x.flatten(1))
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = TinyCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters, device: {device}")


## 3. Autograd

Every tensor remembers the operations that produced it (the "computation graph"). Calling `.backward()` walks that graph in reverse, multiplying local derivatives — *exactly the chain rule from NB 00*, automated.

**This is the single biggest reason PyTorch exists.** Without autograd you'd write 200 lines of conv-backward code per model. With autograd it's one line.


In [ ]:
optim = torch.optim.Adam(model.parameters(), lr=3e-4)
crit  = nn.CrossEntropyLoss()

print("Training loop — note how short the per-step code is:")
print("    for xb, yb in train_loader:")
print("        optim.zero_grad()")
print("        loss = crit(model(xb), yb)")
print("        loss.backward()        # ← autograd does ALL the manual work from NB 02/04")
print("        optim.step()           # ← Adam from NB 06, one call")


## 4. Full training loop end-to-end


In [ ]:
import time
hist = {"epoch": [], "train_loss": [], "test_acc": []}

t0 = time.time()
for ep in range(8):
    model.train()
    epoch_loss = 0; n = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optim.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        optim.step()
        epoch_loss += loss.item() * xb.size(0); n += xb.size(0)
    train_loss = epoch_loss / n
    # Eval
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb.to(device)).argmax(dim=-1)
            correct += (preds.cpu() == yb).sum().item()
            total += len(yb)
    test_acc = correct / total
    hist["epoch"].append(ep+1); hist["train_loss"].append(train_loss); hist["test_acc"].append(test_acc)
    print(f"epoch {ep+1}  loss={train_loss:.3f}  test_acc={test_acc:.3f}")
print(f"\ntotal: {time.time()-t0:.1f}s")


## 5. Plot training curves


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist["epoch"], hist["train_loss"]); ax[0].set_title("train loss"); ax[0].grid(True)
ax[1].plot(hist["epoch"], hist["test_acc"]);   ax[1].set_title("test acc");   ax[1].grid(True)
plt.show()


## 6. Comparison: NumPy vs PyTorch

| Aspect | From-scratch (NB 02–08) | PyTorch (this notebook) |
|---|---|---|
| Lines for MLP forward | ~10 (manual) | ~3 in `nn.Module` |
| Lines for MLP backward | ~15 (manual derivatives) | **0** (autograd) |
| Lines for conv2d | ~25 | ~1 (`nn.Conv2d`) |
| Lines for Adam | ~10 (math) | 1 (`torch.optim.Adam`) |
| GPU support | re-derive everything | `.to('cuda')` |
| Mini-batching | manual loop | `DataLoader` |
| Data augmentation | hand-written | `torchvision.transforms` |
| Mixed precision | infeasible | `torch.cuda.amp` decorator |
| Save / load model | pickle dict | `torch.save / load` |

**When to drop down to manual code:**
- Teaching / understanding (this whole series).
- Custom layer types not in `nn.*`.
- Research papers with novel ops.
- Constrained edge environments without PyTorch.

For 99% of real applications: use PyTorch, but **knowing what it's doing under the hood is the difference between using it and debugging it.**

## 7. Applying this to the project

The project's actual classifier ([src/ccdp/models/damage_classifier.py](../../src/ccdp/models/damage_classifier.py)) and trainer ([src/ccdp/train/train_damage_classifier.py](../../src/ccdp/train/train_damage_classifier.py)) use the *same primitives* you saw in this notebook:
- `torchvision.models.resnet50(pretrained=True)` — pre-built ResNet50 (the 50-layer residual network from NB 04, scaled up).
- `nn.BCEWithLogitsLoss(pos_weight=...)` — multi-label loss with per-class weighting.
- `torch.optim.AdamW` — Adam + decoupled weight decay (the modern default).
- Two-stage fine-tuning — freeze the backbone, train the head, then unfreeze.

You can now read that code and understand every line.
